In [1]:
import sys
print(sys.executable)

C:\Users\SRIKAR\Desktop\VS CODE\rag\.venv\Scripts\python.exe


In [14]:
###Document Loader

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

# Load all the text files from the directory
loader = PyMuPDFLoader("../data/H2405013847.pdf")

documents=loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    """
    Split LangChain documents into smaller chunks.
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Display an example chunk
    if split_docs:
        print("\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [15]:
chunks = split_documents(documents)

print(chunks)

Split 10 documents into 52 chunks

Example Chunk:
Content: IOSR Journal of Computer Engineering (IOSR-JCE)  
e-ISSN: 2278-0661,p-ISSN: 2278-8727, Volume 24, Issue 5, Ser. I (Sep. –Oct. 2022), PP 38-47 
www.iosrjournals.org 
DOI: 10.9790/0661-2405013847       ...
Metadata: {'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20221103070609', 'source': '../data/H2405013847.pdf', 'file_path': '../data/H2405013847.pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': '', 'author': 'Research', 'subject': '', 'keywords': '', 'moddate': 'D:20221103070609', 'trapped': '', 'modDate': 'D:20221103070609', 'creationDate': 'D:20221103070609', 'page': 0}
[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20221103070609', 'source': '../data/H2405013847.pdf', 'file_path': '../data/H2405013847.pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': '', 'author': 'Research', 'subj

In [6]:
chunks = split_documents(documents)

print(chunks)

[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20221103070609', 'source': '../data/H2405013847.pdf', 'file_path': '../data/H2405013847.pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': '', 'author': 'Research', 'subject': '', 'keywords': '', 'moddate': 'D:20221103070609', 'trapped': '', 'modDate': 'D:20221103070609', 'creationDate': 'D:20221103070609', 'page': 0}, page_content='IOSR Journal of Computer Engineering (IOSR-JCE)  \ne-ISSN: 2278-0661,p-ISSN: 2278-8727, Volume 24, Issue 5, Ser. I (Sep. –Oct. 2022), PP 38-47 \nwww.iosrjournals.org \nDOI: 10.9790/0661-2405013847                                  www.iosrjournals.org                                            38 | Page  \nText Detection and Object Recognition from Scene Images \nUsing CNN and YOLOv3 \n \nKaushikDas1, Arun KumarBaruah2 \n \n1(Computer Science and Engineering, Dibrugarh University, India) \n2(Department of Mathematics, Dibrugarh Univ

In [5]:
###Embedding and Vector DB imports

# Numerical computations and vector operations
import numpy as np

# Converts text into embeddings (encoding)
from sentence_transformers import SentenceTransformer

# ChromaDB vector database
import chromadb

# ChromaDB configuration settings
from chromadb.config import Settings

# Generates unique IDs for each document chunk
import uuid

# Type hints for better code readability
from typing import List, Dict, Any, Tuple

# Calculates semantic similarity between embeddings
from sklearn.metrics.pairwise import cosine_similarity

import os

In [3]:
###Embeddings code

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)

            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """

        # Check whether the embedding model is loaded
        if not self.model:
            raise ValueError("Model not loaded")

        # Display how many text chunks are going to be encoded
        print(f"Generating embeddings for {len(texts)} texts...")

        # Encode all text chunks into dense vector embeddings
        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        # Display the shape of the generated embeddings
        print(f"Generated embeddings with shape: {embeddings.shape}")

        # Return the embeddings as a NumPy array
        return embeddings

## initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


C:\Users\SRIKAR\Desktop\VS CODE\rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SRIKAR\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6897.54it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\SRIKAR\AppData\Local\Temp\ipykernel_23072\82644915.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [6]:
###Vector DB code
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory where the vector database is stored.
        """

        # Store the collection name
        self.collection_name = collection_name

        # Store the directory where ChromaDB will persist its data
        self.persist_directory = persist_directory

        # Initialize the ChromaDB client (currently None)
        self.client = None

        # Initialize the ChromaDB collection (currently None)
        self.collection = None

        # Automatically initialize the vector store
        self._initialize_store()
    def _initialize_store(self):
        """
        Initialize the ChromaDB client and collection.
        """

        try:
            # Create the directory for storing the vector database
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create a persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get the existing collection or create a new one
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            # Display initialization details
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain Document objects.
            embeddings: Corresponding embeddings for the documents.
        """

        # Ensure every document has a corresponding embedding
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        # Lists that will be inserted into ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        # Process each document and its embedding
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Generate a unique ID for the document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Store the document text
            documents_text.append(doc.page_content)

            # Convert NumPy array to Python list
            embeddings_list.append(embedding.tolist())

        # Add everything to ChromaDB
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} documents to vector store"
            )

            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore=VectorStore()

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [19]:
# Convert the chunks into plain text
texts = [doc.page_content for doc in chunks]

# Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

# Store documents and embeddings in ChromaDB
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 52 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.12it/s]

Generated embeddings with shape: (52, 384)
Adding 52 documents to vector store...
Successfully added 52 documents to vector store
Total documents in collection: 52


In [21]:
###RAG Retriever
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """

        # Store the VectorStore object
        self.vector_store = vector_store

        # Store the EmbeddingManager object
        self.embedding_manager = embedding_manager

    def retrieve(self,query: str,top_k: int = 5,score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: User search query.
            top_k: Number of top similar documents to retrieve.
            score_threshold: Minimum similarity score required.

        Returns:
            List of dictionaries containing retrieved documents and metadata.
        """

        # Display the query information
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Generate embedding for the user query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            # Search the ChromaDB vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Store retrieved documents
            retrieved_docs = []

            # Check whether documents were found
            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                # Process each retrieved document
                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):

                    # Convert cosine distance into similarity score
                    similarity_score = 1 - distance

                    # Apply similarity threshold
                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [22]:
rag_retriever


In [23]:
rag_retriever.retrieve("What is YOLO V3")

Retrieving documents for query: 'What is YOLO V3'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 13.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_c510f69a_14',
  'content': 'Text Detection and Object Recognition from Scene Images Using CNN and YOLOv3 \nDOI: 10.9790/0661-2405013847                                  www.iosrjournals.org                                            40 | Page  \n \nFig. 1 CNN architecture \n \n2.1.2 \nYOLO V3 \n \nYOLO V3 algorithm is a real-time detection algorithm proposed by Joseph Redmon & Ali Farhadi in \n2018 (Zhang, 2021), it was based on regression technique. It is a CNN, which can predict the position and \ncategory of the multiple target frames simultaneously. It is an improvement of YOLO V1. It utilizes the residual \nneural network as the basic network of the feature extraction, on this premise, a convolution layer is added for \nthe prediction of images of three various scales to get higher semantic data. Furthermore, taking into account the \nclass labels, the YOLO V3 utilizes the logistics rather than the softmax classifier. It uses the FPN network to',
  'metadata': {'conte

In [50]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

# Load environment variables (.env file)
load_dotenv()

ragAPI=""

# Initialize the Groq LLM
llm = ChatGroq(
    groq_api_key=ragAPI,
    model_name="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)


def rag_simple(query: str, retriever, llm, top_k: int = 3):
    """
    Simple RAG Pipeline

    Args:
        query: User's question.
        retriever: RAGRetriever object.
        llm: ChatGroq language model.
        top_k: Number of relevant chunks to retrieve.

    Returns:
        Answer generated by the LLM.
    """

    # Retrieve the most relevant document chunks
    results = retriever.retrieve(
        query=query,
        top_k=top_k
    )

    # Combine retrieved chunks into a single context
    context = (
        "\n\n".join([doc["content"] for doc in results])
        if results else ""
    )

    # If no context is found, return a message
    if not context:
        return "No relevant context found to answer the question."

    # Create the prompt
    prompt = """
Use the following context to answer the question concisely.

Context:
{context}

Question:
{query}

Answer:
"""

    # Send prompt to the Groq LLM
    response = llm.invoke(
        prompt.format(
            context=context,
            query=query
        )
    )

    # Return only the generated answer
    return response.content

In [52]:
answer = rag_simple(
    "What is YOLO V3",
    rag_retriever,
    llm
)

print(answer)

Retrieving documents for query: 'What is YOLO V3'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


YOLO V3 is a real‑time object‑detection algorithm (proposed by Redmon & Farhadi in 2018) that uses a convolutional neural network to simultaneously predict the positions and classes of multiple objects in an image. It improves on YOLO V1 by employing a residual‑network backbone, adding convolution layers for predictions at three different scales, using logistic regression for class scores instead of softmax, and incorporating a Feature Pyramid Network (FPN) to detect objects of various sizes.
